In [0]:
%sql
-- CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_and_Elaprase_table AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE, PRESCRIBER_NPI as NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Specified AS (
    SELECT *
    FROM MPSII_Diagnoses_Specified
    where patient_id in (select distinct a.patient_id from MPSII_Diagnoses_Specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE, PRESCRIBER_NPI as NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Unspecified AS (
    SELECT *
    FROM MPSII_Diagnoses_Unspecified
    where patient_id in (select distinct a.patient_id from MPSII_Diagnoses_Unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
-- MPSII_Treatment_All AS (
--     SELECT * FROM (
--         SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI as NPI
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--     ) t
-- ),
MPSII_Treatment_Elaprase_Only AS (
    SELECT * FROM (
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI as NPI
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT *
    from MPSII_Treatment_Elaprase_Only t  where patient_id in (select distinct a.patient_id from Patients_2Dx_Specified as a)
    --FROM Patients_2Dx_Specified p 
    --INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT * FROM Patients_2Dx_Specified_With_Treatment
    UNION
    --SELECT * FROM Patients_Incremental_Unspecified
    select PATIENT_ID, NPI from MPSII_Diagnoses_Unspecified where patient_id in (select distinct a.patient_id from Patients_Incremental_Unspecified as a) 
),
-- ----------------------------------------------------------
-- Claims universes (5y, 3y, 2y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
-- all_dx_claims_5yr AS (
--     SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE IN ('E761','E763')
--       AND TRANSACTION_STATUS = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- ),
all_tx_claims_5yr AS (
    SELECT DISTINCT PATIENT_ID, BILLING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Mx' as claim_type
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Px'
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, BILLING_NPI AS NPI, SERVICE_DATE AS FILL_DATE,'Mx'
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('J1743')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_claims_5yr AS (
    -- SELECT *, 'DX Claim' FROM all_dx_claims_5yr
    -- UNION
    SELECT *, 'Tx Claim' FROM all_tx_claims_5yr
)

SELECT * from all_claims_5yr
--WHERE claim_type = 'Mx'
